# CHALLENGE 2: Classifying Imagined Movement Across 20 Subjects

## Course architecture

Data acquisition ➡️ Training an ML model ➡️ Visualizing results ➡️ Documenting research

**Optional Dataset Challenge — Colab 2 flavor (multi-subject ML at scale).** This notebook mirrors Lab Stage 2's structure exactly, using a different open dataset: the **[EEG Motor Movement/Imagery Dataset](https://openneuro.org/datasets/ds004362/versions/1.0.0)**, published on OpenNeuro (originally from PhysioNet) — 109 human subjects, 64-channel scalp EEG, recorded while performing and imagining left/right fist movements. Instead of a FABRIC-trained CNN classifying game condition, you'll train a CNN that classifies **which hand a subject imagined moving**, from EEG alone.

## Instructions for the student

As in Lab Stage 2, you act as a **Workflow Designer**, passing each "AI Prompt Challenge" to an AI coding assistant. This is an **optional** extension for anyone who wants a larger multi-subject dataset (109 available subjects vs. Lab Stage 2's 28) and practice with a real, well-documented data-quality quirk.

> ✅ **Ask → Check → Build → Document.** Before running anything an AI tool gives you, check it with three questions: Does it do what I asked? Does it do anything I didn't ask for, such as downloading or deleting files or sending data anywhere? Can I explain each step in plain words? If any answer is no, ask the AI to explain or fix it first. Never paste identifiable participant data, passwords or access keys into a public AI tool.

***
## Module 1: Data Acquisition

We use `mne`'s built-in loader for this exact dataset (the same underlying recordings OpenNeuro ds004362 publishes in BIDS format). Each subject contributes several EEG "runs"; runs 4, 8, and 12 are the **imagined** left-fist/right-fist movement runs we'll use.

**Setup (given):**

In [ ]:
!pip install mne --quiet

import numpy as np
import pandas as pd
import mne
from mne.datasets import eegbci

IMAGERY_RUNS = [4, 8, 12]  # imagined left-fist / right-fist movement runs
EVENT_ID = {"left_fist": 2, "right_fist": 3}

def load_subject_epochs(subject_num, runs=IMAGERY_RUNS):
    """Downloads and epochs one subject's imagined-movement EEG runs."""
    raw_fnames = eegbci.load_data(subject_num, runs, path="eegbci_data/")
    raws = [mne.io.read_raw_edf(f, preload=True, verbose=False) for f in raw_fnames]
    raw = mne.concatenate_raws(raws)
    eegbci.standardize(raw)
    raw.set_montage("standard_1005", on_missing="ignore")
    events, _ = mne.events_from_annotations(raw, verbose=False)
    epochs = mne.Epochs(raw, events, event_id=EVENT_ID, tmin=0, tmax=2,
                         baseline=None, preload=True, verbose=False)
    return epochs

#### Student AI prompt challenge

Copy and paste this text block into your AI assistant:

> "Write code that defines a list called subjects containing 20 subject numbers from 80 to 99 (inclusive) to loop over."


In [ ]:
# TODO: Your code here

In [ ]:
print(len(subjects))

Read in every subject's imagined-movement epochs. Each subject loads on its own without complaint:

In [ ]:
all_epochs = {}
for s in subjects:
    all_epochs[s] = load_subject_epochs(s)
print(f"Loaded {len(all_epochs)} subjects")

But combining every subject's trials into one array for training does **not** go smoothly:

In [ ]:
# Stack every subject's trials into one array (trials x channels x time)
X_all = np.concatenate([ep.get_data() for ep in all_epochs.values()])  # this raises an error

#### Student AI prompt challenge

Copy and paste this text block into your AI assistant:

> "My dictionary all_epochs maps subject number to an MNE Epochs object, and np.concatenate fails because the trials don't all have the same number of time points. Write code that prints each subject's sampling rate (epochs.info['sfreq']) and number of time points, and lists any subjects whose sampling rate differs from the most common one."


In [ ]:
# TODO: Your code here: check sfreq for each subject in `subjects`

#### Student AI prompt challenge

Copy and paste this text block into your AI assistant:

> "Modify the list of subjects to exclude every subject whose sampling rate differs from the most common one, keep everyone else, and print which subjects were excluded and why."


In [ ]:
# TODO: Your code here

In [ ]:
all_epochs = {s: load_subject_epochs(s) for s in subjects}
print(f"Successfully loaded {len(all_epochs)} subjects")

***
## Module 2: Training an ML Model

We train a convolutional neural network to predict which hand a subject imagined moving, directly from their 64-channel EEG epoch.

#### Student AI prompt challenge

Copy and paste this text block into your AI assistant:

> "I have a dictionary all_epochs mapping subject number to an MNE Epochs object, each with a 64-channel x time array of EEG data per trial, labeled left_fist or right_fist. Write pytorch code that builds a single array of all trials across all subjects, trains a simple 1D CNN (convolving across time, 64 input channels) to classify left_fist vs right_fist, converts the EEG from volts to microvolts (multiply by 1e6) before training, holds out 3 subjects entirely for validation (not just trials) using a fixed random seed of 42, prints which subjects were held out, and tracks training/validation loss and accuracy per epoch."


In [ ]:
# TODO: Your code here

***
## Module 3: Visualizing Results

#### Student AI prompt challenge

Copy and paste this text block into your AI assistant:

> "Plot the saved train and validation losses and accuracies per epoch. Then, for the held-out validation subjects, print the final validation accuracy next to the 50% you would get by guessing, and show a confusion matrix of true versus predicted hand (left fist, right fist)."


In [ ]:
# TODO: Your code here

***
## Module 4: Documenting Research

#### Student AI prompt challenge

Copy and paste this text block into your AI assistant:

> "Save the train and validation losses and accuracies per epoch to a .csv file, and add columns recording the held-out validation subjects, the excluded subjects, and the final validation accuracy."


In [ ]:
# TODO: Your code here

***
## Module 5: Additional Prompt Challenges

If time allows, push further with AI-assisted experimentation:

1. **Executed vs. Imagined:** Runs 3, 7, and 11 are the same task, but *executed* rather than imagined. Train the same CNN on executed movement instead — does the model do better or worse than on imagined movement?
2. **Add a Third Class:** Runs 5, 9, 13 add a "both fists vs. both feet" imagery condition. Extend the model to 4-class classification (left fist, right fist, both fists, both feet).
3. **Cross-Subject Generalization:** Instead of holding out 3 random subjects, hold out 3 subjects whose data was recorded on a different day/session than the rest (check the dataset's participant metadata) — does validation accuracy change?
4. **Channel Importance:** Use AI to identify which of the 64 EEG channels contribute most to classification accuracy (e.g. by zeroing out one channel at a time and measuring the accuracy drop).
